Звантажити та відкрити датасет: Individual Household Electric Power Consumption Dataset

In [1]:
import pandas as pd
df_power = pd.read_csv('household_power_consumption.txt', sep=';', low_memory=False)

print("Датасет відкрито перші 5 рядків:")
display(df_power.head())

Датасет відкрито перші 5 рядків:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


Здійснити data cleaning

In [2]:
import numpy as np

df_power.replace('?', np.nan, inplace=True)

df_power = df_power.dropna()

cols_to_float = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_power[cols_to_float] = df_power[cols_to_float].astype(float)

df_power['Time'] = pd.to_datetime(df_power['Time'], format='%H:%M:%S').dt.time

print(f"Дані очищено. Розмір датафрейму тепер= {df_power.shape}")
display(df_power.head())

Дані очищено. Розмір датафрейму тепер= (2049280, 9)


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


Вибірка: активна споживана потужність > 5 кВт

In [3]:
import timeit

def filter_active_power(df):
    return df[df['Global_active_power'] > 5.0]

exec_time_1 = timeit.timeit(lambda: filter_active_power(df_power), number=1)

result_power = filter_active_power(df_power)

print(f"Час виконання процедури: {exec_time_1:.5f} секунд")
print(f"Кількість знайдених записів: {len(result_power)}")
display(result_power.head())

Час виконання процедури: 0.08234 секунд
Кількість знайдених записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


Вибірка: сила струму 19-20 А та порівняння приладів

In [4]:
def filter_intensity_and_appliances(df):
    intensity_filtered = df[(df['Global_intensity'] >= 19.0) & (df['Global_intensity'] <= 20.0)]
    
    result = intensity_filtered[intensity_filtered['Sub_metering_2'] > intensity_filtered['Sub_metering_3']]
    return result

exec_time_2 = timeit.timeit(lambda: filter_intensity_and_appliances(df_power), number=1)
result_appliances = filter_intensity_and_appliances(df_power)
print(f"Час виконання процедури: {exec_time_2:.5f} секунд")
print(f"Кількість знайдених записів: {len(result_appliances)}")
display(result_appliances.head())

Час виконання процедури: 0.00921 секунд
Кількість знайдених записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


Випадкова вибірка 500000 записів

In [5]:
def random_sample_averages(df):
    sample_df = df.sample(n=500000, replace=False, random_state=42)
    averages = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return averages

exec_time_3 = timeit.timeit(lambda: random_sample_averages(df_power), number=1)

result_averages = random_sample_averages(df_power)

print(f"Час виконання процедури: {exec_time_3:.5f} секунд")
print("Середні величини усіх 3-х груп споживання електричної енергії:")
display(result_averages.to_frame(name='Середнє значення'))

Час виконання процедури: 3.75197 секунд
Середні величини усіх 3-х груп споживання електричної енергії:


,Середнє значення
Sub_metering_1,1.119258
Sub_metering_2,1.308912
Sub_metering_3,6.452950


Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.

In [6]:
import datetime
def filter_evening_and_slices(df):
    time_limit = datetime.time(18, 0, 0)
    step1_df = df[(df['Time'] > time_limit) & (df['Global_active_power'] > 6.0)]
    step2_df = step1_df[(step1_df['Sub_metering_2'] > step1_df['Sub_metering_1']) & (step1_df['Sub_metering_2'] > step1_df['Sub_metering_3'])]
    half_index = len(step2_df) // 2
    first_half = step2_df.iloc[:half_index]
    second_half = step2_df.iloc[half_index:]
    
    res_first = first_half.iloc[::3]
    res_second = second_half.iloc[::4]
    final_result = pd.concat([res_first, res_second])
    return final_result

exec_time_4 = timeit.timeit(lambda: filter_evening_and_slices(df_power), number=1)
result_complex = filter_evening_and_slices(df_power)

print(f"Час виконання процедури: {exec_time_4:.5f} секунд")
print(f"Кількість знайдених записів після всіх фільтрів та зрізів: {len(result_complex)}")
display(result_complex.head(10))

Час виконання процедури: 3.61582 секунд
Кількість знайдених записів після всіх фільтрів та зрізів: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0
17504,28/12/2006,21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0
17507,28/12/2006,21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0
17510,28/12/2006,21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0
17513,28/12/2006,21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0
18952,29/12/2006,21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0


Нормалізація та стандартизація

In [7]:
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

df_numeric = df_power[numeric_cols].copy()

df_normalized = (df_numeric - df_numeric.min()) / (df_numeric.max() - df_numeric.min())

df_standardized = (df_numeric - df_numeric.mean()) / df_numeric.std()
print("Пронормований датасет:")
display(df_normalized.head())

print("\nСтандартизований датасет :")
display(df_standardized.head())

Пронормований датасет:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387



Стандартизований датасет :


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955076,2.610720,-1.851816,3.098788,-0.182337,-0.051274,1.249420
1,4.037084,2.770405,-2.225274,4.133799,-0.182337,-0.051274,1.130897
2,4.050325,3.320431,-2.330213,4.133799,-0.182337,0.120487,1.249420
3,4.063566,3.355916,-2.191323,4.133799,-0.182337,-0.051274,1.249420
4,2.434881,3.586572,-1.592555,2.513781,-0.182337,-0.051274,1.249420


Кореляція Пірсона та Спірмена

In [9]:
!pip install scipy

   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
    --------------------------------------- 0.8/36.5 MB 9.8 MB/s eta 0:00:04
   ---- ----------------------------------- 4.2/36.5 MB 14.3 MB/s eta 0:00:03
   ----------- ---------------------------- 10.2/36.5 MB 20.5 MB/s eta 0:00:02
   ---------------- ----------------------- 14.9/36.5 MB 21.4 MB/s eta 0:00:02
   --------------------- ------------------ 19.9/36.5 MB 21.9 MB/s eta 0:00:01
   --------------------------- ------------ 25.2/36.5 MB 22.4 MB/s eta 0:00:01
   --------------------------------- ------ 30.1/36.5 MB 22.7 MB/s eta 0:00:01
   -------------------------------------- - 35.4/36.5 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 22.3 MB/s  0:00:01



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
attr1 = 'Global_active_power'
attr2 = 'Global_intensity'

pearson_corr = df_power[attr1].corr(df_power[attr2], method='pearson')
spearman_corr = df_power[attr1].corr(df_power[attr2], method='spearman')

print(f"Аналіз атрибутів: '{attr1}' та '{attr2}'\n")
print(f"Коефіцієнт кореляції Пірсона: {pearson_corr:.5f}")
print(f"Коефіцієнт кореляції Спірмена: {spearman_corr:.5f}")

Аналіз атрибутів: 'Global_active_power' та 'Global_intensity'

Коефіцієнт кореляції Пірсона: 0.99889
Коефіцієнт кореляції Спірмена: 0.99537


One Hot Encoding категоріального атрибута

In [11]:
import pandas as pd
df_power['Date'] = pd.to_datetime(df_power['Date'], format='%d/%m/%Y')
df_power['Day_of_week'] = df_power['Date'].dt.day_name()

ohe_encoded = pd.get_dummies(df_power['Day_of_week'], prefix='Day', dtype=int)
df_final = pd.concat([df_power[['Date', 'Day_of_week']], ohe_encoded], axis=1)

print("One Hot Encoding для атрибута 'Day_of_week':")
display(df_final.head(10))

One Hot Encoding для атрибута 'Day_of_week':


,Date,Day_of_week,Day_Friday,Day_Monday,Day_Saturday,Day_Sunday,Day_Thursday,Day_Tuesday,Day_Wednesday
0,2006-12-16,Saturday,0,0,1,0,0,0,0
1,2006-12-16,Saturday,0,0,1,0,0,0,0
2,2006-12-16,Saturday,0,0,1,0,0,0,0
3,2006-12-16,Saturday,0,0,1,0,0,0,0
4,2006-12-16,Saturday,0,0,1,0,0,0,0
5,2006-12-16,Saturday,0,0,1,0,0,0,0
6,2006-12-16,Saturday,0,0,1,0,0,0,0
7,2006-12-16,Saturday,0,0,1,0,0,0,0
8,2006-12-16,Saturday,0,0,1,0,0,0,0
9,2006-12-16,Saturday,0,0,1,0,0,0,0
